In [ ]:
#from IPython.core.display import display, HTML
#display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
# Replace the following with your path
%cd "/path/to/your/folder"

In [ ]:
# Load modules
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load .env
load_dotenv()

In [ ]:
# Replace the following with your set up
os.environ['API_ENDPOINT']='XX'
os.environ['API_DEPLOYMENT']='XX'
os.environ['API_VERSION']='XX'
os.environ['API_KEY']='XX' # keep in confidential

client = AzureOpenAI(azure_endpoint=os.getenv('API_ENDPOINT'), api_version=os.getenv('API_VERSION'),
                     api_key=os.getenv('API_KEY'))

In [ ]:
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string

def preprocess_text(text):
    # Replace 'N/A' with an empty string
    text = text.replace('N/A', '')
    # Convert to lowercase
    text = text.lower()
    # Remove extra spaces
    text = ' '.join(text.split())
    return text

def cosine_distance(text1, text2):
    # Preprocess the texts
    text1 = preprocess_text(text1)
    text2 = preprocess_text(text2)
    
    # Create the CountVectorizer and transform the texts (remove punctuations but keep operators)
    vectorizer = CountVectorizer(token_pattern=r'\b\w+\b|[<>]').fit_transform([text1, text2])
    # Compute cosine similarity between the two texts
    similarity = cosine_similarity(vectorizer)[0][1]
    # Return cosine distance (1 - similarity)
    return 1 - similarity

def jaccard_distance(text1, text2):
    # Preprocess the texts
    text1 = preprocess_text(text1)
    text2 = preprocess_text(text2)
    
    # Create the CountVectorizer and transform the texts (remove punctuations but keep operators)
    vectorizer = CountVectorizer(token_pattern=r'\b\w+\b|[<>]').fit_transform([text1, text2])
    
    # Convert the sparse matrix to dense and get the binary occurrence of tokens
    binary_occurrences = vectorizer.toarray().astype(bool)
    
    # Calculate Jaccard distance
    intersection = np.logical_and(binary_occurrences[0], binary_occurrences[1]).sum()
    union = np.logical_or(binary_occurrences[0], binary_occurrences[1]).sum()
    
    return 1 - (intersection / union if union != 0 else 0)

In [ ]:
# Function to split and concatenate attribute categorized and cutoffs
def split_and_concatenate(row):
    if row["Attribute categorized"] == "Not applicable" and row["Cutoffs"] == "Not applicable":
        return "Not applicable"
    if row["Cutoffs"] == "Not reported":
        return "Not reported"
    
    attributes = row["Attribute categorized"].split(";")
    cutoffs = row["Cutoffs"].split(";")
    if len(attributes) == len(cutoffs):
        # Concatenate corresponding elements
        return "; ".join(["{} : {}".format(attr.strip(), cutoff.strip()) for attr, cutoff in zip(attributes, cutoffs)])
    return "Mismatch in attribute and cutoff counts"


In [ ]:
def process_nctid(nctid):
    import os
    from openai import AzureOpenAI
    from dotenv import load_dotenv
    import csv
    import numpy as np
    import json
    import pandas as pd
    from openpyxl import load_workbook, Workbook
    from openpyxl.styles import Font, Alignment
    from openpyxl.utils.dataframe import dataframe_to_rows
    import time

    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    from scipy.spatial.distance import jaccard
    
    # Initialize token tracking list
    token_tracking = []
    
    def add_token_record(nctid, task_name, input_tokens, output_tokens, processing_time):
        """Add a token usage record to the tracking list"""
        token_tracking.append({
            'NCT_ID': nctid,
            'Task': task_name,
            'Input_Tokens': input_tokens,
            'Output_Tokens': output_tokens,
            'Total_Tokens': input_tokens + output_tokens,
            'Processing_Time_Seconds': processing_time
        })
    
    def save_token_tracking(token_tracking_data):
        """Save token tracking data to Excel file"""
        # Create DataFrame from tracking data
        df_tokens = pd.DataFrame(token_tracking_data)
        
        # Excel file path for token tracking
        token_workbook_path = "processing_cost_cutoff.xlsx"
        
        # Check if the workbook file exists
        if os.path.exists(token_workbook_path):
            # Load the existing workbook
            workbook = load_workbook(filename=token_workbook_path)
            if 'Token_Usage' not in workbook.sheetnames:
                sheet = workbook.create_sheet('Token_Usage')
            else:
                sheet = workbook['Token_Usage']
                
            # Get the next empty row
            next_row = sheet.max_row + 1 if sheet.max_row > 1 else 1
            
            # Write the header row if it's the first entry
            if next_row == 1:
                for col_num, column_title in enumerate(df_tokens.columns, 1):
                    cell = sheet.cell(row=1, column=col_num)
                    cell.value = column_title
                    cell.font = Font(bold=True)
            
            # Write the data rows
            for row in dataframe_to_rows(df_tokens, index=False, header=False):
                sheet.append(row)
                
        else:
            # Create a new workbook
            workbook = Workbook()
            sheet = workbook.active
            sheet.title = 'Token_Usage'
            
            # Write the header row
            for col_num, column_title in enumerate(df_tokens.columns, 1):
                cell = sheet.cell(row=1, column=col_num)
                cell.value = column_title
                cell.font = Font(bold=True)
            
            # Write the data rows
            for row in dataframe_to_rows(df_tokens, index=False, header=False):
                sheet.append(row)
        
        # Set column widths
        for col in sheet.columns:
            sheet.column_dimensions[col[0].column_letter].width = 15
        
        # Save the workbook
        workbook.save(token_workbook_path)
        print(f"Token tracking saved to: {token_workbook_path}")
    
    def task1(prompt1_link):
    
        # 1.1 Construct prompt
        with open(prompt1_link, 'r') as file:
            prompt1 = file.read()

        prompt2 = "# Input" + '\n' + 'The outcome information from the ClinicalTrials.gov registration is:'

        prompt = prompt1 + "\n" + prompt2 + "\n" + outcome_text + "\n" + prompt1 
        ##print(prompt)

        # 1.2 Run prompt
        message = [{'role': 'user', 'content': prompt}]
        start_time = time.time()  # Start timing

        #response=client.chat.completions.create(model='gpt-4o-2024-08-06', 
        response=client.chat.completions.create(model=os.getenv('API_DEPLOYMENT'), 
                                                messages=message, 
                                                seed=66,
                                                max_tokens=4096,
                                                temperature=0,
                                                top_p=1
                                                )

        end_time = time.time()  # End timing
        elapsed_time = end_time - start_time
        print(f"API processing time for {nctid}: {elapsed_time:.2f} seconds")
        
        # Track token usage for Task 1
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        add_token_record(nctid, "Task 1 - Parse Elements", input_tokens, output_tokens, elapsed_time)
    
        # 1.3 Save results
        content = response.choices[0].message.content
        table_data = json.loads(content.strip("```json"))
        columns = table_data["table"]["columns"]
        rows = table_data["table"]["rows"]
        output = pd.DataFrame(rows, columns=columns)
        
        return output

    ## Task 1: Parse elements (prospective) ##

    out_link=f"out_pro/{nctid}_merged.json"

    # Information from clinicaltrials.gov
    with open(out_link, 'r') as file:
        data = json.load(file)

    outcome_text = str(data)

    # Convert the JSON data to a DataFrame
    df1 = pd.DataFrame(data)

    df1['Version'] = "1"

    ## Task 2: Parse elements (last) ##

    out_link=f"out_last_cut/{nctid}_merged.json"

    # Information from clinicaltrials.gov
    with open(out_link, 'r') as file:
        data = json.load(file)

    outcome_text = str(data)

    # Convert the JSON data to a DataFrame
    df2 = pd.DataFrame(data)

    df2['Version'] = "2"

    merged_set = pd.concat([df1, df2], axis=0)
    merged_set = merged_set[['Version'] + [col for col in merged_set.columns if col != 'Version']]
    
    merged_set_list = merged_set.values.tolist()

    outcome_text = str(merged_set_list)

    prompt1_link = 'Prompts/Define_Compare_Other elements/Task1_cut.txt'
    output1 = task1(prompt1_link)

    # Merge outcome text with output
    merged_df = pd.merge(merged_set, output1, on=['Version', 'Identifier'], how='inner')  # 'how' can be 'inner', 'left', 'right', or 'outer' depending on your needs

    output1["Cutoff"] = output1.apply(split_and_concatenate, axis=1)

    output1['Number of outcomes defined by cutoffs'] = output1['Number of outcomes defined by cutoffs'].astype(str)
    output1.loc[(output1['Number of outcomes defined by cutoffs'] != "1") & (output1['Number of outcomes defined by cutoffs'] != "Not applicable"), 'Cutoff'] = 'Not valid'

    # Save the results
    result1 = pd.merge(output1, merged_set, on=['Version', 'Identifier'], how='outer')

    # Add nctid, link
    inf_link=f"inf_pro/{nctid}.json"

    with open(inf_link, 'r') as file:
        data = json.load(file)
    v1=data['Version Before Recruiting']+1
    v2=data['Most Recent Version']+1

    result1['index'] = result1.index + 1
    result1['record_id'] = nctid
    result1['out_link'] = f'https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#outcome-measures-card'
    result1['res_link'] = f"https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#results-outcome-measures-card"
    ##print(result1)

    # 6.3 Save results to a local file

    # Workbook file path
    workbook_path = "cutoff_definition.xlsx"

    # Check if the workbook file exists
    if os.path.exists(workbook_path):
        # Load the existing workbook
        workbook = load_workbook(filename=workbook_path)
        if 'Comparisons' not in workbook.sheetnames:
            sheet = workbook.create_sheet('Comparisons')
        else:
            sheet = workbook['Comparisons']
    else:
        # Create a new workbook
        workbook = Workbook()
        sheet = workbook.active
        sheet.title = 'Comparisons'


    # Get the next empty row
    next_row = sheet.max_row + 1 if sheet.max_row > 1 else 1

    # Write the header row if it's the first entry
    if next_row == 1:
        for col_num, column_title in enumerate(result1.columns, 1):
            cell = sheet.cell(row=1, column=col_num)
            cell.value = column_title
            cell.font = Font(bold=True)

    # Write the data rows
    for row in dataframe_to_rows(result1, index=False, header=False):
        sheet.append(row)

    # Adjust cell size and text wrapping
    for row in sheet.iter_rows(min_row=next_row, max_row=sheet.max_row):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True)

    # Set column widths to a fixed value
    fixed_width = 20  # Specify the desired fixed width
    for col in sheet.columns:
        sheet.column_dimensions[col[0].column_letter].width = fixed_width

    # Save the workbook
    workbook.save(workbook_path)



    ## Task 3: Import matching results ##

    task3 = pd.read_csv(f"{nctid}_table3.csv")

    # GPT might mistakenly enter [Missing outcome] for [Added outcome], and vice versa
    task3['Outcome (prospective)'] = task3['Outcome (prospective)'].replace('[Missing outcome]', '[Added outcome]')
    task3['Outcome (last)'] = task3['Outcome (last)'].replace('[Added outcome]', '[Missing outcome]')

    task3['Identifier (prospective)'] = task3['Identifier (prospective)'].replace('[Missing outcome]', '[Added outcome]')
    task3['Identifier (last)'] = task3['Identifier (last)'].replace('[Added outcome]', '[Missing outcome]')

    # Split into two DataFrames based on the Version column
    pros = result1[result1['Version'] == "1"]  # Rows where Version == "1"
    last = result1[result1['Version'] == "2"]  # Rows where Version == "2"

    # 4.1 Combine output tables from tasks 1, 2, and 3

    # Add prospective/last to each variable name
    table1 = pros.rename(columns=lambda x: f"{x} (prospective)")
    table2 = last.rename(columns=lambda x: f"{x} (last)")

    # Merge the tables from 1 and 2 and the matched table from 3
    combined_table = pd.merge(table1, task3, on='Identifier (prospective)', how='outer')
    combined_table = pd.merge(combined_table, table2, on='Identifier (last)', how='outer')

    #combined_table.to_csv(f'{nctid}_measined.csv', index=False, encoding='utf-8-sig')

    # 4.3 Keep only the matched outcomes

    # Create a boolean mask to identify the rows to keep
    mask = ~ ((combined_table['Outcome (last)'] == '[Missing outcome]') | pd.isna(combined_table['Outcome (last)']) |
    (combined_table['Outcome (prospective)'] == '[Added outcome]') | pd.isna(combined_table['Outcome (prospective)']))

    # Filter the DataFrame to keep only the desired rows
    matched = combined_table[mask]
      
    
    if not matched.empty:
        # 3.1.1 Extract original texts

        out_link=f"out_pro/{nctid}_merged.json"

        with open(out_link, 'r') as file:
            data = json.load(file)

        df1 = pd.DataFrame({'Original text (prospective)': data})


        # Create identifier
        def extract_priority_and_number(data):
            identifier = data.get('Identifier', '')
            return identifier.strip()

        # Apply the function to create new column
        df1['Identifier (prospective)'] = df1['Original text (prospective)'].apply(extract_priority_and_number).str.capitalize()

            # 3.2.1 Extract original texts

        out_link=f"out_last_cut/{nctid}_merged.json"

        with open(out_link, 'r') as file:
            data = json.load(file)

        df2 = pd.DataFrame({'Original text (last)': data})

        # Add the Identifier column
        df2['Identifier (last)'] = df2['Original text (last)'].apply(extract_priority_and_number).str.capitalize()


        # 4.4 Add original texts

        combined_table_tx = pd.merge(matched, df1, on='Identifier (prospective)', how='inner')
        combined_table_tx = pd.merge(combined_table_tx, df2, on='Identifier (last)', how='inner')


        # Specify the columns you want to keep
        columns_to_keep = ['Identifier (prospective)', 'Cutoff (prospective)', 'Identifier (last)',
                           'Cutoff (last)']

        # Create a new DataFrame with only the specified columns

        matched_string = combined_table_tx.loc[:, columns_to_keep].to_json(orient='records')

        # 5.1  Construct prompt
        with open("Prompts/Define_Compare_Other elements/Task5_cut_simple.txt", 'r') as file:
            prompt1 = file.read()

        prompt2="# Input" + "\n" + "Here is the combined table GPT made with the presence and definitions in both the prospective and last versions:"
        prompt3=matched_string


        prompt = prompt1 + "\n" + prompt2 + "\n" + prompt3 + "\n" + prompt1

        ##print(prompt)

        # 5.2 Run prompt

        message = [{'role': 'user', 'content': prompt}]
        start_time = time.time()  # Start timing
        
        #response4 = client.chat.completions.create(model='gpt-4o-2024-08-06',  
        response4 =client.chat.completions.create(model=os.getenv('API_DEPLOYMENT'), 
                                                   messages=message, 
                                                   seed=66,
                                                   max_tokens=4096,
                                                   temperature=0,
                                                   top_p=1)
        end_time = time.time()  # End timing
        elapsed_time = end_time - start_time
        print(f"API processing time for {nctid}: {elapsed_time:.2f} seconds")
        
        # Track token usage for Task 5
        input_tokens = response4.usage.prompt_tokens
        output_tokens = response4.usage.completion_tokens
        add_token_record(nctid, "Task 5 - Compare Outcomes", input_tokens, output_tokens, elapsed_time)
    
        ##print(response4)

        # 5.3 Save results to a local file

        # Extract the content part from the OpenAIObject
        content4 = response4.choices[0].message.content

        # Parse the content string as JSON
        table_data4 = json.loads(content4.strip("```json"))

        # Extract the table columns and rows
        columns = table_data4["table"]["columns"]
        rows = table_data4["table"]["rows"]

        # Create a DataFrame from the rows and columns
        table4 = pd.DataFrame(rows, columns=columns)

        #print(table4)

        # Combine with the tables from tasks 1,2,3
        result = pd.merge(table4, combined_table, on=['Identifier (prospective)','Identifier (last)'], how='outer')


        # Desired column order
        column_order = [
            "Identifier (prospective)", "Outcome (prospective)", "Identifier (last)", "Outcome (last)",
             "Cutoff (prospective)", "Cutoff (last)", 
            "Rationale, Change status", "Change status"
        ]

        # Reorder the DataFrame columns
        result3 = result.reindex(columns=column_order)

        result3.to_csv(f"{nctid}_cut.csv", index=False, encoding='utf-8-sig')



        # Save the results


        ##print(result)

        # Add nctid, link
        inf_link=f"inf_pro/{nctid}.json"

        with open(inf_link, 'r') as file:
            data = json.load(file)
        v1=data['Version Before Recruiting']+1
        v2=data['Most Recent Version']+1

        result3['index'] = result3.index + 1
        result3['record_id'] = nctid
        result3['out_link'] = f'https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#outcome-measures-card'
        result3['res_link'] = f"https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#results-outcome-measures-card"
        ##print(result3)

        # 6.3 Save results to a local file

        # Workbook file path
        workbook_path = "cutoff_change_status.xlsx"

        # Check if the workbook file exists
        if os.path.exists(workbook_path):
            # Load the existing workbook
            workbook = load_workbook(filename=workbook_path)
            if 'Comparisons' not in workbook.sheetnames:
                sheet = workbook.create_sheet('Comparisons')
            else:
                sheet = workbook['Comparisons']
        else:
            # Create a new workbook
            workbook = Workbook()
            sheet = workbook.active
            sheet.title = 'Comparisons'


        # Get the next empty row
        next_row = sheet.max_row + 1 if sheet.max_row > 1 else 1

        # Write the header row if it's the first entry
        if next_row == 1:
            for col_num, column_title in enumerate(result3.columns, 1):
                cell = sheet.cell(row=1, column=col_num)
                cell.value = column_title
                cell.font = Font(bold=True)

        # Write the data rows
        for row in dataframe_to_rows(result3, index=False, header=False):
            sheet.append(row)

        # Adjust cell size and text wrapping
        for row in sheet.iter_rows(min_row=next_row, max_row=sheet.max_row):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True)

        # Set column widths to a fixed value
        fixed_width = 20  # Specify the desired fixed width
        for col in sheet.columns:
            sheet.column_dimensions[col[0].column_letter].width = fixed_width

        # Save the workbook
        workbook.save(workbook_path)
        
    else:
        print("No matched outcomes.")
        unmatched_df = combined_table[~mask]
        result = unmatched_df.loc[:, ['Identifier (prospective)','Outcome (prospective)','Identifier (last)','Outcome (last)']]
        
        # Save the final comparison
        result3 = result

        ##print(result)

        # Add nctid, link
        inf_link=f"inf_pro/{nctid}.json"

        with open(inf_link, 'r') as file:
            data = json.load(file)
        v1=data['Version Before Recruiting']+1
        v2=data['Most Recent Version']+1

        result3['index'] = result3.index + 1
        result3['record_id'] = nctid
        result3['out_link'] = f'https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#outcome-measures-card'
        result3['res_link'] = f"https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#results-outcome-measures-card"
        ##print(result3)

        # 6.3 Save results to a local file

        # Workbook file path
        workbook_path = "cutoff_change_status.xlsx"

        # Check if the workbook file exists
        if os.path.exists(workbook_path):
            # Load the existing workbook
            workbook = load_workbook(filename=workbook_path)
            if 'Comparisons' not in workbook.sheetnames:
                sheet = workbook.create_sheet('Comparisons')
            else:
                sheet = workbook['Comparisons']
        else:
            # Create a new workbook
            workbook = Workbook()
            sheet = workbook.active
            sheet.title = 'Comparisons'


        # Get the next empty row
        next_row = sheet.max_row + 1 if sheet.max_row > 1 else 1

        # Write the header row if it's the first entry
        if next_row == 1:
            for col_num, column_title in enumerate(result3.columns, 1):
                cell = sheet.cell(row=1, column=col_num)
                cell.value = column_title
                cell.font = Font(bold=True)

        # Write the data rows
        for row in dataframe_to_rows(result3, index=False, header=False):
            sheet.append(row)

        # Adjust cell size and text wrapping
        for row in sheet.iter_rows(min_row=next_row, max_row=sheet.max_row):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True)

        # Set column widths to a fixed value
        fixed_width = 20  # Specify the desired fixed width
        for col in sheet.columns:
            sheet.column_dimensions[col[0].column_letter].width = fixed_width

        # Save the workbook
        workbook.save(workbook_path)

    # Save token tracking data at the end of processing
    save_token_tracking(token_tracking)

In [ ]:
import time
# Read NCTIDs from a text file
with open('nctid_new150.txt', 'r') as file:
    nctids = [line.strip() for line in file if line.strip()]

# Process each NCTID
# Initialize a counter variable
n = 1

for nctid in nctids:
    print(f"Processing NCTID {n}/{len(nctids)}: {nctid}")
    process_nctid(nctid)
    print(f"Finished processing NCTID: {nctid}")
    
    # Optional: Add a delay between processing each NCTID to avoid rate limiting
    time.sleep(5)
    
    # Increment the counter
    n += 1

print("All NCTIDs processed.")

In [ ]:
import sys

# Stop the script
sys.exit()

In [ ]:
# Prepare for REDCAp import